In [13]:
import numpy as np


In [14]:
#dummy words :-

X = np.array([
    [1, 0, 1, 0],   # word 1 embedding
    [0, 2, 0, 2],   # word 2 embedding
    [1, 1, 1, 1]    # word 3 embedding
])

print("Input embeddings (X):\n", X)


Input embeddings (X):
 [[1 0 1 0]
 [0 2 0 2]
 [1 1 1 1]]


In [15]:
# Create random weight matrices for Q, K, V
# These are learned during training in real models.

W_Q = np.random.randn(4, 4)
W_K = np.random.randn(4, 4)
W_V = np.random.randn(4, 4)

print("W_Q:\n", W_Q)
print("\nW_K:\n", W_K)
print("\nW_V:\n", W_V)


W_Q:
 [[-0.43222317 -1.37247558 -1.25798725 -0.05870823]
 [ 0.48126287 -1.0846131   0.47715778  1.18448601]
 [-0.08442858 -0.18398198 -0.18644514 -0.24075296]
 [-0.31410941  0.37845547  0.44677017 -1.72145584]]

W_K:
 [[-1.59715721  0.63793625  0.87523989 -1.44271725]
 [ 1.09424777  2.39261754 -0.61998486  0.06581375]
 [-1.55414076  1.2230108  -0.39271726 -1.66792368]
 [ 0.49189819  0.2605691  -0.94678188  0.7754748 ]]

W_V:
 [[-0.48238066  0.92254797  1.33131344  0.19616516]
 [-0.66708246  0.85438643  0.25884046  0.09839897]
 [ 0.81338916  0.16519166 -0.40053547 -1.38823683]
 [-0.93472908  0.25474873 -0.30763483 -1.16643966]]


In [16]:
# Compute Query, Key, and Value vectors
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)


Q:
 [[-0.51665175 -1.55645756 -1.44443239 -0.29946119]
 [ 0.33430692 -1.41231526  1.84785589 -1.07393965]
 [-0.34949828 -2.26261519 -0.52050444 -0.83643102]]

K:
 [[-3.15129798  1.86094705  0.48252263 -3.11064093]
 [ 3.17229192  5.30637329 -3.13353349  1.68257711]
 [-1.56515201  4.5141337  -1.08424411 -2.26935237]]

V:
 [[ 0.3310085   1.08773964  0.93077797 -1.19207167]
 [-3.20362309  2.21827032 -0.09758875 -2.13608138]
 [-1.27080304  2.1968748   0.8819836  -2.26011236]]


In [17]:
# Compute raw attention scores using dot product of Q and K^T
# Q = Query vectors for each word
# K.T = Key vectors for each word (transposed)
# The result shows how much each word should pay attention to every other word.

scores = Q @ K.T

print("Raw attention scores:\n", scores)


Raw attention scores:
 [[ -1.03381659  -5.87580429  -3.97171873]
 [  0.55052827 -14.03105743  -6.46500044]
 [ -0.75855245 -12.891333    -7.20421895]]


In [18]:
# Scale the raw attention scores by sqrt(d_k)
# This prevents very large values that would break softmax.

dk = K.shape[-1]        # d_k = dimension of key vectors (here = 4)
scores_scaled = scores / np.sqrt(dk)

print("Scaled scores:\n", scores_scaled)


Scaled scores:
 [[-0.51690829 -2.93790214 -1.98585936]
 [ 0.27526414 -7.01552871 -3.23250022]
 [-0.37927623 -6.4456665  -3.60210948]]


In [19]:
# Softmax function for converting scores to probabilities
def softmax(x):
    # subtract max for numerical stability (avoids overflow)
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Apply softmax to scaled scores
attn_weights = softmax(scores_scaled)

print("Attention Weights (Softmax applied):\n", attn_weights)


Attention Weights (Softmax applied):
 [[7.58150071e-01 6.73489625e-02 1.74500966e-01]
 [9.70265613e-01 6.61514757e-04 2.90728720e-02]
 [9.59544135e-01 2.22569235e-03 3.82301726e-02]]


In [20]:
# Final self-attention output
# Multiply attention weights with V to get the new contextual representation

output = attn_weights @ V

print("Self-Attention Output:\n", output)


Self-Attention Output:
 [[-0.18656293  1.35742486  0.85300387 -1.44202388]
 [ 0.28210103  1.12073325  0.9286791  -1.22374716]
 [ 0.26190397  1.13265828  0.92662372 -1.23500412]]


In [21]:
# Let's simulate a second attention head using new random weights

# New weight matrices for head 2
W_Q2 = np.random.randn(4, 4)
W_K2 = np.random.randn(4, 4)
W_V2 = np.random.randn(4, 4)

# Compute Q, K, V for head 2
Q2 = X @ W_Q2
K2 = X @ W_K2
V2 = X @ W_V2

# Attention scores for head 2
scores2 = Q2 @ K2.T / np.sqrt(dk)

# Softmax to get attention weights for head 2
attn_w2 = softmax(scores2)

# Final output for head 2
output2 = attn_w2 @ V2

print("Head 1 Output:\n", output)
print("\nHead 2 Output:\n", output2)

# Multi-head output = concatenation of all head outputs
multi_head_output = np.concatenate([output, output2], axis=-1)

print("\nMulti-Head Output (Concatenated):\n", multi_head_output)


Head 1 Output:
 [[-0.18656293  1.35742486  0.85300387 -1.44202388]
 [ 0.28210103  1.12073325  0.9286791  -1.22374716]
 [ 0.26190397  1.13265828  0.92662372 -1.23500412]]

Head 2 Output:
 [[ 3.26851418  2.98920402  5.42846616 -0.31547327]
 [ 3.32274803  3.01142176  5.49351423 -0.28826227]
 [ 3.33226804  3.01506919  5.50470168 -0.28318645]]

Multi-Head Output (Concatenated):
 [[-0.18656293  1.35742486  0.85300387 -1.44202388  3.26851418  2.98920402
   5.42846616 -0.31547327]
 [ 0.28210103  1.12073325  0.9286791  -1.22374716  3.32274803  3.01142176
   5.49351423 -0.28826227]
 [ 0.26190397  1.13265828  0.92662372 -1.23500412  3.33226804  3.01506919
   5.50470168 -0.28318645]]


In [22]:
# Final summary of this notebook
print("""
Summary of Notebook 1 — Transformer Basics

1. We created simple embeddings (X) to represent words.
2. We made random weight matrices (W_Q, W_K, W_V) like in real Transformers.
3. We computed Q, K, V vectors using matrix multiplication.
4. We computed raw attention scores using Q @ K^T.
5. We scaled the scores by sqrt(d_k) for stability.
6. We applied softmax to turn scores into attention weights.
7. We multiplied attention weights with V to get the final contextual output.
8. We created multiple attention heads and saw they produce different outputs.
9. We concatenated head outputs → multi-head attention output.

This is exactly how real Transformers like BERT, DistilBERT, GPT-2, LLaMA, etc. work internally.
""")



Summary of Notebook 1 — Transformer Basics

1. We created simple embeddings (X) to represent words.
2. We made random weight matrices (W_Q, W_K, W_V) like in real Transformers.
3. We computed Q, K, V vectors using matrix multiplication.
4. We computed raw attention scores using Q @ K^T.
5. We scaled the scores by sqrt(d_k) for stability.
6. We applied softmax to turn scores into attention weights.
7. We multiplied attention weights with V to get the final contextual output.
8. We created multiple attention heads and saw they produce different outputs.
9. We concatenated head outputs → multi-head attention output.

This is exactly how real Transformers like BERT, DistilBERT, GPT-2, LLaMA, etc. work internally.

